# Step 1 -- Player frame table + ball frame table (named tracking)

One row per **(frame, person)** for the 25 people, and one row per **frame** for the ball / carrier / possession.
Everything in the pressure analysis reads these two Parquet files.

Inputs (all in `new_cache`): `barca_atletico_match_cache.pkl` (named players, teams), the tracked ball, the v10 carrier
(named ids), the homography. Nothing is re-detected or re-tracked here.

What is added on top of the raw boxes:
- **pitch position in metres** (feet = bottom-centre of the box, through the per-frame homography) and an
  **out-of-pitch / no-homography** flag
- **smoothed position, velocity, speed, acceleration** (Savitzky-Golay per *continuous segment* of a player --
  never smoothed or differentiated across a gap where he was not visible)
- **attack direction per team**, found from the goalkeepers' side (not hard-coded), and `x_att` = distance from the
  team's own goal line along its attack direction (0 = own goal, 105 = opponent goal) -- the same meaning for both teams
- carrier, possession, stoppage, ball position and distance to the ball on every row
- `source` (tracked / recovered / interpolated / auto) and `segment_id`, so any metric can exclude estimated boxes

**Single broadcast camera:** a player who is not in the picture has *no row* (he is not standing still). Per-player
totals must be read per **visible minute**; `visible_s` is in the metadata.

In [ ]:
import os, io, json, pickle
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from collections import Counter
from scipy.signal import savgol_filter

CACHE_DIR = r"C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache"
MATCH_CACHE_PATH  = os.path.join(CACHE_DIR, "barca_atletico_match_cache.pkl")
BALL_TRACKED_PATH = os.path.join(CACHE_DIR, "barca_atletico_first_half_ball_tracked_cache.pkl")
CARRIER_PATH      = os.path.join(CACHE_DIR, "barca_atletico_first_half_ball_carrier_named.pkl")
HOMOGRAPHY_PATH   = os.path.join(CACHE_DIR, "homography_cache_barca_atletico_firsthalf.pkl")

OUT_PLAYER_TABLE = os.path.join(CACHE_DIR, "player_frame_table_named.parquet")
OUT_BALL_TABLE   = os.path.join(CACHE_DIR, "ball_frame_table_named.parquet")
OUT_META         = os.path.join(CACHE_DIR, "match_meta_named.json")
OUT_FIG_DIR      = os.path.join(CACHE_DIR, "frame_table_checks")

FPS = 25
PX_PER_METER = 10.0
PITCH_LENGTH, PITCH_WIDTH = 105.0, 68.0
OUT_OF_PITCH_MARGIN_M = 5.0       # feet further than this outside the pitch -> pitch position not trusted
SMOOTH_WINDOW = 11                # Savitzky-Golay window (frames, odd) = 0.44 s
SMOOTH_POLY = 2
MAX_SEGMENT_GAP_FRAMES = 1        # a missing frame starts a new segment (no smoothing / velocity across it)
MAX_PLAUSIBLE_SPEED_MPS = 10.5    # above this = tracking / homography artefact (flagged, not removed)
MAX_PLAUSIBLE_BALL_SPEED_MPS = 40.0
STOPPAGE_GAP_SEC = 10.0           # same rule as the pressure notebook: no carrier for >= 10 s = dead ball


class _Compat(pickle.Unpickler):  # pickles written with numpy 2 (Kaggle) read with numpy 1
    def find_class(self, module, name):
        if module.startswith("numpy._core"):
            module = module.replace("numpy._core", "numpy.core", 1)
        return super().find_class(module, name)


def load_pickle(path):
    with open(path, "rb") as fh:
        blob = fh.read()
    try:
        return pickle.loads(blob)
    except ModuleNotFoundError:
        return _Compat(io.BytesIO(blob)).load()


M = load_pickle(MATCH_CACHE_PATH)
TC, PINFO, LOCKED, TEAM = M["tracking_cache"], M["player_info"], M["locked_class_by_id"], M["team"]
BALL = load_pickle(BALL_TRACKED_PATH)
CARRIER = load_pickle(CARRIER_PATH)
HC = load_pickle(HOMOGRAPHY_PATH)
TEAM_NAMES = {int(k): v for k, v in TEAM["team_names"].items()}          # e.g. {0: 'ATM', 1: 'BAR'}
NAME_TO_TEAM = {v: k for k, v in TEAM_NAMES.items()}
N_FRAMES = max(TC) + 1
os.makedirs(OUT_FIG_DIR, exist_ok=True)
print(f"frames {N_FRAMES} | people {len(PINFO)} {dict(Counter(LOCKED.values()))} | teams {TEAM_NAMES}")
print(f"ball {len(BALL)} frames | carrier {len(CARRIER)} frames | homography {len(HC)}")

## 1. Raw rows: one per (frame, person), pitch position of the feet

In [ ]:
def H_at(f):
    H = HC[f] if isinstance(HC, list) and 0 <= f < len(HC) else (HC.get(f) if isinstance(HC, dict) else None)
    return None if H is None else np.asarray(H, np.float64)


rows = []
for f in range(N_FRAMES):
    tr = TC.get(f, {}).get("tracks", [])
    if not tr:
        continue
    B = np.asarray([t["bbox"] for t in tr], dtype=np.float64)
    feet = np.stack([(B[:, 0] + B[:, 2]) / 2, B[:, 3]], 1)
    H = H_at(f)
    P = (cv2.perspectiveTransform(feet[None].astype(np.float32), H).reshape(-1, 2) / PX_PER_METER
         if H is not None else np.full((len(tr), 2), np.nan))
    for k, t in enumerate(tr):
        rows.append((f, int(t["track_id"]), t.get("class"), t.get("team"), t.get("team_name"), t.get("name"),
                     t.get("number"), B[k, 0], B[k, 1], B[k, 2], B[k, 3], float(t.get("conf", np.nan)),
                     t.get("source", "tracked"), t.get("assign_score"), feet[k, 0], feet[k, 1], P[k, 0], P[k, 1],
                     len(t.get("duplicate_boxes", ()))))

pf = pd.DataFrame(rows, columns=["frame_idx", "track_id", "role", "team", "team_name", "name", "number",
                                 "x1", "y1", "x2", "y2", "conf", "source", "assign_score", "foot_px_x", "foot_px_y",
                                 "pitch_x_raw", "pitch_y_raw", "n_duplicate_boxes"])
del rows
pf["team"] = pd.to_numeric(pf["team"], errors="coerce").astype("Int8")
pf["number"] = pd.to_numeric(pf["number"], errors="coerce").astype("Int16")
pf["assign_score"] = pd.to_numeric(pf["assign_score"], errors="coerce")
m = OUT_OF_PITCH_MARGIN_M
pf["pitch_valid"] = (pf.pitch_x_raw.between(-m, PITCH_LENGTH + m) & pf.pitch_y_raw.between(-m, PITCH_WIDTH + m)).fillna(False)
pf["t_sec"] = pf.frame_idx / FPS
print(f"{len(pf):,} rows | pitch position valid on {pf.pitch_valid.mean():.2%} | sources {pf.source.value_counts().to_dict()}")

## 2. Smoothed position, velocity, speed, acceleration (per continuous segment)
A segment = consecutive frames of one player with a valid pitch position. Savitzky-Golay gives the smoothed position
and its derivatives in one step; segments shorter than the window fall back to a short centred mean / finite
differences, and a 1-2 frame segment gets no velocity at all (NaN) rather than a made-up one.

In [ ]:
pf.sort_values(["track_id", "frame_idx"], inplace=True, ignore_index=True)
for c in ("pitch_x", "pitch_y", "vx", "vy", "ax", "ay"):
    pf[c] = np.nan
pf["segment_id"] = -1

valid_idx = np.where(pf.pitch_valid.values)[0]
tid = pf.track_id.values[valid_idx]
fr = pf.frame_idx.values[valid_idx]
brk = np.r_[True, (tid[1:] != tid[:-1]) | (np.diff(fr) > MAX_SEGMENT_GAP_FRAMES)]
seg = np.cumsum(brk) - 1
starts = np.r_[np.where(brk)[0], len(valid_idx)]

X, Y = pf.pitch_x_raw.values[valid_idx], pf.pitch_y_raw.values[valid_idx]
sx, sy, vx, vy, ax, ay = (np.full(len(valid_idx), np.nan) for _ in range(6))
dt = 1.0 / FPS
for a, b in zip(starts[:-1], starts[1:]):
    n = b - a
    if n >= SMOOTH_WINDOW:
        for src, s0, s1, s2 in ((X, sx, vx, ax), (Y, sy, vy, ay)):
            s0[a:b] = savgol_filter(src[a:b], SMOOTH_WINDOW, SMOOTH_POLY)
            s1[a:b] = savgol_filter(src[a:b], SMOOTH_WINDOW, SMOOTH_POLY, deriv=1, delta=dt)
            s2[a:b] = savgol_filter(src[a:b], SMOOTH_WINDOW, SMOOTH_POLY, deriv=2, delta=dt)
    else:
        for src, s0, s1 in ((X, sx, vx), (Y, sy, vy)):
            s0[a:b] = pd.Series(src[a:b]).rolling(3, center=True, min_periods=1).mean().values
            if n >= 3:
                s1[a:b] = np.gradient(s0[a:b], dt)

pf.loc[valid_idx, "pitch_x"] = sx
pf.loc[valid_idx, "pitch_y"] = sy
pf.loc[valid_idx, "vx"], pf.loc[valid_idx, "vy"] = vx, vy
pf.loc[valid_idx, "ax"], pf.loc[valid_idx, "ay"] = ax, ay
pf.loc[valid_idx, "segment_id"] = seg
pf["speed_mps"] = np.hypot(pf.vx, pf.vy)
# acceleration along the running direction (+ = speeding up, - = braking); undefined when nearly still
pf["accel_mps2"] = np.where(pf.speed_mps > 0.5, (pf.vx * pf.ax + pf.vy * pf.ay) / pf.speed_mps, np.nan)
pf["speed_implausible"] = pf.speed_mps > MAX_PLAUSIBLE_SPEED_MPS
seg_len = pd.Series(seg).value_counts()
print(f"segments: {len(seg_len):,} | median length {seg_len.median():.0f} frames | "
      f"shorter than the smoothing window: {(seg_len < SMOOTH_WINDOW).mean():.1%}")
print(f"rows with a speed: {pf.speed_mps.notna().mean():.1%} | implausible (> {MAX_PLAUSIBLE_SPEED_MPS} m/s): "
      f"{pf.speed_implausible.mean():.2%}")

## 3. Ball frame table: ball, carrier, possession, stoppages
`possession_team` / `carrier_team` are the integer kit teams (same numbering as `team` in the player table);
`*_name` gives BAR / ATM. Ball speed only from consecutive real positions (detected / smoothed), never across a lost gap.

In [ ]:
bt = pd.DataFrame({"frame_idx": np.arange(N_FRAMES)})
bt["t_sec"] = bt.frame_idx / FPS
bpx, bpitch, bsrc, bconf = [], [], [], []
for f in range(N_FRAMES):
    b = BALL.get(f) or {}
    xy, xp = b.get("xy_px"), b.get("xy_pitch")
    bpx.append(xy if xy is not None else (np.nan, np.nan))
    bpitch.append(xp if xp is not None else (np.nan, np.nan))
    bsrc.append(b.get("source", "lost")); bconf.append(b.get("conf"))
bt[["ball_px_x", "ball_px_y"]] = np.asarray(bpx, dtype=float)
bt[["ball_pitch_x", "ball_pitch_y"]] = np.asarray(bpitch, dtype=float)
bt["ball_source"] = bsrc
bt["ball_conf"] = pd.to_numeric(pd.Series(bconf), errors="coerce")
bt.loc[bt.ball_source == "lost", ["ball_px_x", "ball_px_y", "ball_pitch_x", "ball_pitch_y"]] = np.nan

ok = bt.ball_pitch_x.notna().values
P = bt[["ball_pitch_x", "ball_pitch_y"]].values
spd = np.full(N_FRAMES, np.nan)
both = np.r_[False, ok[2:] & ok[:-2], False]
spd[1:-1] = np.where(both[1:-1], np.hypot(P[2:, 0] - P[:-2, 0], P[2:, 1] - P[:-2, 1]) * FPS / 2, np.nan)
bt["ball_speed_mps"] = spd
bt["ball_speed_implausible"] = bt.ball_speed_mps > MAX_PLAUSIBLE_BALL_SPEED_MPS

car = [CARRIER.get(f) or {} for f in range(N_FRAMES)]
bt["carrier_track_id"] = pd.array([c.get("track_id") for c in car], dtype="Int32")
bt["carrier_method"] = [c.get("method") for c in car]
bt["carrier_team_name"] = [c.get("team") for c in car]
bt["possession_team_name"] = [c.get("possession_team") for c in car]
bt["carrier_team"] = pd.array([NAME_TO_TEAM.get(n) for n in bt.carrier_team_name], dtype="Int8")
bt["possession_team"] = pd.array([NAME_TO_TEAM.get(n) for n in bt.possession_team_name], dtype="Int8")
bt["carrier_name"] = [PINFO.get(c, {}).get("name") if c is not None else None for c in bt.carrier_track_id.astype(object)]

# stoppages: no carrier for >= STOPPAGE_GAP_SEC (same rule as the pressure notebook, section 0b)
miss = bt.carrier_track_id.isna().values
run_id = np.cumsum(np.r_[True, miss[1:] != miss[:-1]])
runs = pd.DataFrame({"run": run_id, "miss": miss, "f": bt.frame_idx}).groupby("run").agg(
    miss=("miss", "first"), start=("f", "first"), end=("f", "last"), n=("f", "size"))
stoppages = runs[runs.miss & (runs.n >= STOPPAGE_GAP_SEC * FPS)][["start", "end", "n"]].reset_index(drop=True)
bt["is_stoppage"] = False
for s, e in zip(stoppages.start, stoppages.end):
    bt.loc[s:e, "is_stoppage"] = True

cnt = pf.groupby("frame_idx").size()
bt["n_people_visible"] = bt.frame_idx.map(cnt).fillna(0).astype(int)
for k in TEAM_NAMES:
    c = pf[(pf.team == k) & pf.role.isin(["player", "goalkeeper"])].groupby("frame_idx").size()
    bt[f"n_visible_team{k}"] = bt.frame_idx.map(c).fillna(0).astype(int)
bt["homography_valid"] = [H_at(f) is not None for f in range(N_FRAMES)]
print(f"ball on pitch: {bt.ball_pitch_x.notna().mean():.1%} | carrier: {bt.carrier_track_id.notna().mean():.1%} | "
      f"stoppages: {len(stoppages)} ({stoppages.n.sum() / FPS:.0f} s)")

## 4. Attack direction per team (from the goalkeepers) + per-row context
The goalkeeper of each team stands near his own goal, so his mean pitch x tells which goal each team defends in this
half. `x_att` (0 = own goal line -> 105 = opponent goal line) and `dist_from_own_goal` then mean the same for both teams.

In [ ]:
gk = pf[(pf.role == "goalkeeper") & pf.pitch_x.notna() & pf.team.notna()]
gk_x = gk.groupby("team").pitch_x.mean().to_dict()
ATTACK_DIR = {}
for k in TEAM_NAMES:
    if k in gk_x:
        ATTACK_DIR[k] = 1 if gk_x[k] < PITCH_LENGTH / 2 else -1
if len(ATTACK_DIR) == 2 and ATTACK_DIR[0] == ATTACK_DIR[1]:
    raise ValueError(f"both goalkeepers on the same side ({gk_x}) -- check goalkeeper teams / homography")
for k, d in ATTACK_DIR.items():
    print(f"team {k} ({TEAM_NAMES[k]}): GK mean x {gk_x[k]:.1f} m -> defends the {'left (x=0)' if d == 1 else 'right (x=105)'} "
          f"goal, attacks {'+x' if d == 1 else '-x'}")
if ATTACK_DIR.get(0) == -1:
    print("  NOTE: team 0 attacks -x here. The v30 pressure notebook hard-codes 'team 0 attacks +x' -- the new "
          "analysis uses x_att / attack_dir instead, so it does not depend on the team numbering.")

pf["attack_dir"] = pf.team.map(ATTACK_DIR).astype("Int8")
_dir = pf.attack_dir.astype("float").to_numpy()
pf["x_att"] = np.where(_dir == 1, pf.pitch_x, np.where(_dir == -1, PITCH_LENGTH - pf.pitch_x, np.nan))
pf["dist_from_own_goal"] = pf.x_att.clip(0, PITCH_LENGTH)
pf["third"] = pd.cut(pf.x_att, [-np.inf, 35, 70, np.inf], labels=["own_third", "middle_third", "attacking_third"])

pf = pf.merge(bt[["frame_idx", "carrier_track_id", "possession_team", "is_stoppage", "ball_pitch_x", "ball_pitch_y"]],
              on="frame_idx", how="left")
pf["is_carrier"] = (pf.track_id.to_numpy() == pf.carrier_track_id.astype("float").to_numpy())
pf["in_possession"] = pd.array(np.where(pf.team.isna() | pf.possession_team.isna(), None,
                                        pf.team.astype("float").to_numpy() == pf.possession_team.astype("float").to_numpy()),
                               dtype="boolean")
pf["dist_to_ball_m"] = np.hypot(pf.pitch_x - pf.ball_pitch_x, pf.pitch_y - pf.ball_pitch_y)
pf.drop(columns=["ball_pitch_x", "ball_pitch_y"], inplace=True)
pf.sort_values(["frame_idx", "track_id"], inplace=True, ignore_index=True)
print(f"player table: {len(pf):,} rows x {pf.shape[1]} columns")

## 5. Checks (read these before using the tables)
Expected: top speeds 8-10 m/s, almost nothing above 10.5 m/s, the carrier within ~1-2 m of the ball, both teams
attacking opposite directions, possession share close to the carrier notebook. A high implausible-speed share on
`auto` boxes = auto-assigned pieces that jump between people -> worth reviewing those players.

In [ ]:
live = pf[~pf.is_stoppage]
print("=== visibility per person (minutes on screen, live play) ===")
vis = (live.groupby(["track_id"]).size() / FPS / 60).rename("visible_min")
summ = pd.DataFrame({"name": [PINFO[t].get("name") for t in vis.index],
                     "team": [PINFO[t].get("team") for t in vis.index],
                     "role": [LOCKED.get(t) for t in vis.index], "visible_min": vis.values}, index=vis.index)
v = live[live.speed_mps.notna() & ~live.speed_implausible]
dist = v.groupby("track_id").speed_mps.sum() / FPS
summ["dist_m"] = dist
summ["m_per_visible_min"] = summ.dist_m / (v.groupby("track_id").size() / FPS / 60)
summ["top_speed"] = v.groupby("track_id").speed_mps.quantile(0.995)
summ["implausible_pct"] = live.groupby("track_id").speed_implausible.mean() * 100
print(summ.sort_values(["team", "role", "visible_min"], ascending=[True, True, False]).round(1).to_string())

print("\n=== speed by role (m/s, plausible rows) ===")
print(v.groupby("role").speed_mps.describe(percentiles=[.5, .9, .99]).round(2).to_string())
print("\n=== implausible speed share by box source ===")
print((pf.groupby("source").speed_implausible.mean() * 100).round(2).astype(str).add(" %").to_string())

cr = pf[pf.is_carrier & pf.dist_to_ball_m.notna()]
print(f"\ncarrier rows: {len(cr):,} | carrier -> ball distance median {cr.dist_to_ball_m.median():.2f} m, "
      f"90% {cr.dist_to_ball_m.quantile(.9):.2f} m")
miss_car = bt.carrier_track_id.notna() & ~bt.frame_idx.isin(pf.loc[pf.is_carrier, "frame_idx"])
print(f"carrier frames where the carrier has no row: {miss_car.sum()} ({miss_car.mean():.2%})")
ps = bt.loc[~bt.is_stoppage, "possession_team_name"].value_counts(normalize=True) * 100
print("possession share (live play):", ps.round(1).to_dict())
print(f"stoppages: {len(stoppages)} | ball implausible speed frames: {int(bt.ball_speed_implausible.sum())}")

fig, axs = plt.subplots(1, 3, figsize=(16, 4))
v.speed_mps.clip(upper=12).hist(bins=60, ax=axs[0]); axs[0].set_title("player speed (m/s)")
summ.sort_values("visible_min").visible_min.plot.barh(ax=axs[1], figsize=(16, 6)); axs[1].set_title("visible minutes")
axs[1].set_yticklabels([summ.loc[i, "name"] for i in summ.sort_values("visible_min").index], fontsize=7)
for k, d in ATTACK_DIR.items():
    s = pf[(pf.team == k).fillna(False) & (pf.role == "player") & pf.pitch_x.notna()]
    s = s.sample(min(20000, len(s)), random_state=0)
    axs[2].scatter(s.pitch_x, s.pitch_y, s=1, alpha=0.2, label=f"{TEAM_NAMES[k]} attacks {'+x' if d == 1 else '-x'}")
axs[2].set_xlim(-5, 110); axs[2].set_ylim(-5, 73); axs[2].legend(markerscale=10); axs[2].set_title("positions")
plt.tight_layout(); plt.savefig(os.path.join(OUT_FIG_DIR, "frame_table_checks.png"), dpi=110); plt.show()

## 6. Save tables + metadata

In [ ]:
pf.to_parquet(OUT_PLAYER_TABLE, index=False)
bt.to_parquet(OUT_BALL_TABLE, index=False)
meta = {
    "fps": FPS, "pitch_length": PITCH_LENGTH, "pitch_width": PITCH_WIDTH, "n_frames": int(N_FRAMES),
    "team_names": {str(k): v for k, v in TEAM_NAMES.items()},
    "attack_dir": {str(k): int(v) for k, v in ATTACK_DIR.items()},
    "goalkeeper_mean_x": {str(k): round(float(v), 2) for k, v in gk_x.items()},
    "people": {str(t): {"name": i.get("name"), "number": i.get("number"), "team_name": i.get("team"),
                        "team": (None if i.get("team") is None else NAME_TO_TEAM.get(i.get("team"))),
                        "role": LOCKED.get(t),
                        "visible_s": round(float(vis.get(t, 0) * 60), 1)} for t, i in PINFO.items()},
    "stoppages": stoppages.rename(columns={"start": "start_frame", "end": "end_frame", "n": "n_frames"}).to_dict("records"),
    "params": {"smooth_window": SMOOTH_WINDOW, "smooth_poly": SMOOTH_POLY,
               "max_segment_gap_frames": MAX_SEGMENT_GAP_FRAMES, "out_of_pitch_margin_m": OUT_OF_PITCH_MARGIN_M,
               "max_plausible_speed_mps": MAX_PLAUSIBLE_SPEED_MPS, "stoppage_gap_sec": STOPPAGE_GAP_SEC},
    "columns": {"player": list(pf.columns), "ball": list(bt.columns)},
}
with open(OUT_META, "w", encoding="utf-8") as fh:
    json.dump(meta, fh, indent=1, ensure_ascii=False, default=lambda o: int(o) if isinstance(o, np.integer) else str(o))
print(f"Saved {OUT_PLAYER_TABLE} ({os.path.getsize(OUT_PLAYER_TABLE) / 1e6:.0f} MB, {len(pf):,} rows)")
print(f"Saved {OUT_BALL_TABLE} ({len(bt):,} rows)\nSaved {OUT_META}")
print("\nplayer columns:", list(pf.columns))
print("ball columns:", list(bt.columns))